<a href="https://colab.research.google.com/github/lctung/fontdiffuser-finetune-colab/blob/main/FontDiffuser_finetuning.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 步驟一： 下載助教的 Github 專案 (Fontdiffuser)

---



In [ ]:
!git clone https://github.com/lctung/fontdiffuser-finetune.git

**下載預訓練權重檔**:

---

In [ ]:
%cd /content/fontdiffuser-finetune
# 安裝 gdown（如果尚未安裝）
!pip install -U gdown

# 建立 ckpt 資料夾（如果尚未存在）
!mkdir -p ckpt

# 下載四個檔案
!gdown --id 1UF4nIcL3PRJeQOQOPFFuGzj30v67rlNn -O ckpt/scr_210000.pth
!gdown --id 1XIY1QnEIKYmciFnxJ0r48f2LpS8KfZXh -O ckpt/unet.pth
!gdown --id 1-ywYwsfr8ryE86FgY9Xlub2uhPzZ_WWR -O ckpt/style_encoder.pth
!gdown --id 1xX-yTNXhniBNR9R5sc1v7-Ghd64HsKIo -O ckpt/content_encoder.pth

**下載torch相關函式庫:**

---



# **下面這格程式執行中會需要按一次enter**

In [ ]:
# 安裝 Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils python3.10-dev -y

# 切換默認 python 版本
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 2
!sudo update-alternatives --config python3

# 重新安裝 pip
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

In [ ]:
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117

In [ ]:
import torch
print("GPU 可用:", torch.cuda.is_available())
print("PyTorch 版本:", torch.__version__)


**安裝所需套件**:

---

In [ ]:
!pip install -r /content/fontdiffuser-finetune/requirements.txt

**掛載雲端硬碟空間**:

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**上傳手寫字體壓縮檔&製作訓練資料集合**:

---

In [ ]:
# ----------------上傳手寫字體壓縮檔----------------
%cd /content/fontdiffuser-finetune/split_train_test
from google.colab import files
import zipfile
import os

# 1. 顯示彈跳視窗讓使用者選擇上傳檔案
uploaded = files.upload()

# 2. 假設只上傳一個 zip 檔，取得檔名
for filename in uploaded.keys():
    zip_path = filename
    break
print(filename)

# 3. 解壓縮檔案到目前資料夾
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")  # 解壓縮到目前目錄
print("解壓縮檔案完成")
filename = os.path.splitext(filename)[0]

# ----------------製作訓練資料集合:----------------
%run colab_split_data.py --folder_param "{filename}"

**開始進行微調Fontdiffuser**:

---

In [ ]:
%cd /content/fontdiffuser-finetune
!accelerate launch train.py \
    --seed=123 \
    --experience_name="FontDiffuser_110598080_fineturning" \
    --data_root="data_examples" \
    --output_dir="outputs/FontDiffuser_110598080_fineturning" \
    --report_to="tensorboard" \
    --resolution=96 \
    --style_image_size=96 \
    --content_image_size=96 \
    --content_encoder_downsample_size=3 \
    --channel_attn=True \
    --content_start_channel=64 \
    --style_start_channel=64 \
    --train_batch_size=8 \
    --perceptual_coefficient=0.01 \
    --offset_coefficient=0.5 \
    --max_train_steps=5000 \
    --ckpt_interval=1000 \
    --gradient_accumulation_steps=1 \
    --log_interval=50 \
    --learning_rate=5e-5 \
    --lr_scheduler="linear" \
    --lr_warmup_steps=500 \
    --drop_prob=0.1 \
    --mixed_precision="no" \
    --usecolab